In [ ]:
import json
import glob
import re
import os
from tqdm import tqdm

# extract entity and relation
def parse_triple(triple):
    pattern = re.compile(r"\[entity1, ([^\]]+)\]: (.+?), \[relation, ([^\]]+)\]: (.+?), \[entity2, ([^\]]+)\]: (.+)")
    match = pattern.search(triple["triples"])
    if match:
        entity1_id, entity1, relation_id, relation, entity2_id, entity2 = match.groups()
        return (entity1_id, entity1.strip()), (relation_id, relation.strip()), (entity2_id, entity2.strip())
    else:
        raise ValueError("Triple format is incorrect")

def extract_entities_and_relations(file_path):
    entities = set()
    relations = set()
    entity_relation_map = {}
    lines = []
    
    with open(file_path, 'r') as file:
        lines = file.readlines()
        for line in lines:
            try:
                data = json.loads(line)
                entity1, relation, entity2 = parse_triple(data)
                
                entities.add(entity1)
                entities.add(entity2)
                relations.add(relation)
                
                if entity1 not in entity_relation_map:
                    entity_relation_map[entity1] = []
                entity_relation_map[entity1].append((relation, entity2))
                
                if entity2 not in entity_relation_map:
                    entity_relation_map[entity2] = []
                entity_relation_map[entity2].append((relation, entity1))
                
            except (ValueError, KeyError, json.JSONDecodeError) as e:
                print(f"Error processing line: {line.strip()} - {e}")
    
    return entities, relations, entity_relation_map, lines

#######################old extract_entities_and_relations, works ver1#################################
# def extract_entities_and_relations(file_path):
#     entities = set()
#     relations = set()
#     entity_relation_map = {}
    
#     with open(file_path, 'r') as file:
#         lines = file.readlines()
#         for line in lines:
#             try:
#                 data = json.loads(line)
#                 entity1, relation, entity2 = parse_triple(data)
                
#                 entities.add(entity1)
#                 entities.add(entity2)
#                 relations.add(relation)
                
#                 if entity1 not in entity_relation_map:
#                     entity_relation_map[entity1] = []
#                 entity_relation_map[entity1].append((relation, entity2))
                
#                 if entity2 not in entity_relation_map:
#                     entity_relation_map[entity2] = []
#                 entity_relation_map[entity2].append((relation, entity1))
                
#             except (ValueError, KeyError, json.JSONDecodeError) as e:
#                 print(f"Error processing line: {line.strip()} - {e}")
    
#     return entities, relations, entity_relation_map

# def load_all_entities_and_relations(directory_path):
#     all_entities = set()
#     all_relations = set()
#     all_entity_relation_map = {}
    
#     for file_path in glob.glob(f"{directory_path}/*.jsonl"):
#         entities, relations, entity_relation_map = extract_entities_and_relations(file_path)
#         all_entities.update(entities)
#         all_relations.update(relations)
        
#         for entity, relations in entity_relation_map.items():
#             if entity not in all_entity_relation_map:
#                 all_entity_relation_map[entity] = []
#             all_entity_relation_map[entity].extend(relations)
    
#     return all_entities, all_relations, all_entity_relation_map
#######################extract_entities_and_relations, ver1#################################

def build_hops(entity, entity_relation_map, depth=1, max_depth=None, path=None, visited=None, cache=None):
    if path is None:
        path = [entity]
    if visited is None:
        visited = set()
    if cache is None:
        cache = {}

    if max_depth and depth > max_depth:
        return []
    if entity not in entity_relation_map:
        return []
    
    cache_key = (entity, depth)
    if cache_key in cache:
        return cache[cache_key]

    hops = []
    visited.add(entity)
    
    for relation, linked_entity in entity_relation_map[entity]:
        if linked_entity not in visited:
            new_path = path + [relation, linked_entity]
            if depth == max_depth:
                hop = format_hop(new_path)
                hops.append(hop)
            else:
                sub_hops = build_hops(linked_entity, entity_relation_map, depth+1, max_depth, new_path, visited.copy(), cache)
                if sub_hops:
                    hops.extend(sub_hops)
                else:
                    hop = format_hop(new_path)
                    hops.append(hop)
            break  
    
    cache[cache_key] = hops
    return hops

#######################old build hops, works ver1#################################
# def build_hops(entity, entity_relation_map, depth=1, max_depth=5, path=None, visited=None, cache=None):
#     if path is None:
#         path = [entity]
#     if visited is None:
#         visited = set()
#     if cache is None:
#         cache = {}

#     if depth > max_depth or entity not in entity_relation_map:
#         return []
    
#     cache_key = (entity, depth)
#     if cache_key in cache:
#         return cache[cache_key]

#     hops = []
#     visited.add(entity)
    
#     for relation, linked_entity in entity_relation_map[entity]:
#         if linked_entity not in visited:
#             new_path = path + [relation, linked_entity]
#             if depth == max_depth:
#                 hop = format_hop(new_path)
#                 hops.append(hop)
#             else:
#                 sub_hops = build_hops(linked_entity, entity_relation_map, depth+1, max_depth, new_path, visited.copy(), cache)
#                 if sub_hops:
#                     hops.extend(sub_hops)
#             break  
    
#     cache[cache_key] = hops
#     return hops
#######################build hops, ver1#################################

def format_hop(path):
    hop_str = "hops: "
    entity_counter = 1
    for i in range(0, len(path), 2):
        entity = path[i]
        relation = path[i+1] if i+1 < len(path) else None
        if i > 0:
            hop_str += ", "
        hop_str += f"[entity{entity_counter}, {entity[0]}]: {entity[1]}"
        entity_counter += 1
        if relation:
            hop_str += f", [relation, {relation[0]}]: {relation[1]}"
    return hop_str

def process_files(input_directory, output_directory):

    os.makedirs(output_directory, exist_ok=True)
    
    input_files = glob.glob(f"{input_directory}/*.jsonl")
    total_files = len(input_files)


    for input_file_path in tqdm(input_files, desc="Processing files", unit="file"):
      
        entities, relations, entity_relation_map, lines = extract_entities_and_relations(input_file_path)
        
        file_name = os.path.basename(input_file_path)
        base_name = os.path.splitext(file_name)[0].replace('reformed', '')
        output_file_path = os.path.join(output_directory, f"{base_name}_hops.jsonl")
        
        generated_hops = set()  
        
        with open(output_file_path, 'w') as output_file:
            for line in lines:
                try:
                    data = json.loads(line)
                    entity1, _, entity2 = parse_triple(data)
                    
                    for entity in [entity1, entity2]:
                        if entity not in generated_hops:
                            all_hops = build_hops(entity, entity_relation_map, depth=1, max_depth=5)
                            if all_hops:
                      
                                all_hops_depths = [hop for hop in all_hops if len(hop.split('entity')) - 1 == 5]
                                if all_hops_depths:
                  
                                    output_file.write(json.dumps({"hop": all_hops_depths[0]}) + '\n')
                                else:
                                
                                    output_file.write(json.dumps({"hop": all_hops[-1]}) + '\n')
                            generated_hops.add(entity) 
                except (ValueError, KeyError, json.JSONDecodeError) as e:
                    print(f"Error processing line: {line.strip()} - {e}")


#######################old process file, works ver1#################################
# def process_files(input_directory, output_directory):
#  
#     os.makedirs(output_directory, exist_ok=True)
    
#     input_files = glob.glob(f"{input_directory}/*.jsonl")
#     total_files = len(input_files)

#    
#     for input_file_path in tqdm(input_files, desc="Processing files", unit="file"):
#        
#         entities, relations, entity_relation_map = extract_entities_and_relations(input_file_path)
        
#         
#         file_name = os.path.basename(input_file_path)
#         base_name = os.path.splitext(file_name)[0].replace('reformed', '')
#         output_file_path = os.path.join(output_directory, f"{base_name}_hops.jsonl")
        
#        
#         with open(output_file_path, 'w') as output_file:
#             for entity in entities:
#                 hops = build_hops(entity, entity_relation_map, depth=1, max_depth=5)
#                 for hop in hops:
#                     output_file.write(json.dumps({"hop": hop}) + '\n')
#######################process file, ver1#################################

input_directory = '/hkfs/home/project/hk-project-p00201316/st_st176945/entity_rels_reformed'  
output_directory = '/hkfs/home/project/hk-project-p00201316/st_st176945/generated_hops'  
process_files(input_directory, output_directory)